# RescueVision - FloodNet PNG tile pack (CPU)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
# pin albumentations: newer versions (preinstalled on the GPU image) have a
# stricter shape check that rejects our verified-consistent (HxWx3, HxW) pairs;
# --upgrade forces the downgrade past pip's "already satisfied"; all transforms
# ALSO use is_check_shapes=False as a version-proof second layer
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_workers=0: DataLoader workers died on the Kaggle GPU image; pin_memory
# off too (useless with workers=0, and a hard-kill vector on some drivers)
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

# per-epoch heartbeat: every trainer that logs an epoch also lands in the marks,
# so a silent kill still tells us how far training got and with what metrics
import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np, pandas as pd, random
from PIL import Image
from pathlib import Path

ROOT = None
for _c in Path("/kaggle/input").rglob("class_mapping.csv"):
    ROOT = _c.parent
    break
if ROOT is None and Path("/tmp/floodnet_src").exists():
    ROOT = Path("/tmp/floodnet_src")          # synthetic smoke path
assert ROOT is not None, "FloodNet dataset not found under /kaggle/input"
print("FloodNet root:", ROOT)
mark("floodnet:root", extra=str(ROOT))
TILES = Path("/tmp/floodnet_tiles")
(TILES / "train").mkdir(parents=True, exist_ok=True)
(TILES / "val").mkdir(parents=True, exist_ok=True)

from rescuevision.constants import FLOODNET_CLASSES, NUM_FLOODNET_CLASSES

import re
def norm_name(s):
    return re.sub(r"[^a-z]", "", str(s).lower())

NAME_TO_ID = {norm_name(v): k for k, v in FLOODNET_CLASSES.items()}

OFFICIAL_PALETTE = {  # official FloodNet colors, fallback if the CSV can't be parsed
    (0, 0, 0): 0, (72, 0, 255): 1, (255, 0, 0): 2, (0, 255, 255): 3, (255, 255, 0): 4,
    (0, 255, 0): 5, (0, 0, 255): 6, (255, 0, 255): 7, (0, 128, 128): 8, (128, 128, 128): 9,
}

color_to_id = {}
try:
    cm = pd.read_csv(ROOT / "class_mapping.csv")
    print("class_mapping.csv:\n", cm.to_string())
    cols = {c.lower(): c for c in cm.columns}
    name_col = next((cols[c] for c in cols if "name" in c or "class" in c), None)
    has_rgb = all(any(k in cols for k in (key,)) for key in ("r", "g", "b"))
    hex_col = next((cols[c] for c in cols if "hex" in c or "color" in c), None)
    for _, row in cm.iterrows():
        cid = NAME_TO_ID.get(norm_name(row[name_col])) if name_col else None
        if cid is None:
            continue
        if has_rgb:
            color_to_id[(int(row[cols["r"]]), int(row[cols["g"]]), int(row[cols["b"]]))] = cid
        elif hex_col and str(row[hex_col]).startswith("#"):
            h = str(row[hex_col]).lstrip("#")
            color_to_id[tuple(int(h[i:i+2], 16) for i in (0, 2, 4))] = cid
except Exception as e:
    print("could not parse class_mapping.csv:", e)
if len(color_to_id) < NUM_FLOODNET_CLASSES and (ROOT / "class_mapping.csv").exists() is False:
    print("no class_mapping.csv - masks assumed grayscale ids")
elif len(color_to_id) < NUM_FLOODNET_CLASSES:
    print("falling back to official FloodNet palette")
    color_to_id = OFFICIAL_PALETTE

def mask_to_ids(mask_rgb):
    # (H, W, 3) uint8 -> (H, W) class ids, nearest palette color
    m = mask_rgb.astype(np.int16)
    ids = np.zeros(m.shape[:2], dtype=np.uint8)
    best = np.full(m.shape[:2], 3 * 255, dtype=np.int16)
    for color, cid in color_to_id.items():
        d = (np.abs(m[:, :, 0] - color[0]) + np.abs(m[:, :, 1] - color[1])
             + np.abs(m[:, :, 2] - color[2])).astype(np.int16)
        better = d < best
        ids[better] = cid
        np.minimum(best, d, out=best)
    return ids

sample_mask = next((ROOT / "Train/Labeled/Flooded/mask").glob("*.png"))
sm = Image.open(sample_mask)
print("sample mask mode:", sm.mode, sm.size)
grayscale_mode = sm.mode in ("L", "P")
print("grayscale_mode:", grayscale_mode)
mark("floodnet:format", extra=f"grayscale={grayscale_mode}")

In [ ]:
import random, os
from tqdm.auto import tqdm

TILE = 512
SEED = 42
VAL_FRAC = 0.15
MAX_TRAIN_TILES = 16000
MAX_VAL_TILES = 2400
MAX_TMP_GB = 40

img_dirs = [ROOT / "Train/Labeled/Flooded", ROOT / "Train/Labeled/Non-Flooded"]
samples = []
for d in img_dirs:
    for img_path in sorted((d / "image").glob("*.jpg")):
        mask_path = d / "mask" / (img_path.stem + "_lab.png")
        if not mask_path.exists():
            mask_path = d / "mask" / (img_path.stem + ".png")
        if mask_path.exists():
            samples.append((img_path, mask_path))
print(f"{len(samples)} labeled images")
mark("floodnet:samples", extra=str(len(samples)))

rng = random.Random(SEED)
val_set = set(rng.sample(range(len(samples)), max(1, int(len(samples) * VAL_FRAC))))

def tree_bytes(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())

records = []
n_train = n_val = 0
for i, (img_path, mask_path) in enumerate(tqdm(samples, desc="tiling")):
    split = "val" if i in val_set else "train"
    img = np.array(Image.open(img_path).convert("RGB"))
    ids = np.clip(np.array(Image.open(mask_path).convert("L"), dtype=np.uint8), 0, NUM_FLOODNET_CLASSES - 1)
    H, W = img.shape[:2]
    for y in range(0, H - TILE + 1, TILE):
        for x in range(0, W - TILE + 1, TILE):
            tile_ids = ids[y:y+TILE, x:x+TILE]
            if len(np.unique(tile_ids)) < 2:
                continue
            stem = f"{img_path.stem}_{y//TILE}_{x//TILE}"
            img_npy = TILES / split / f"{stem}_img.png"
            ids_npy = TILES / split / f"{stem}_ids.png"
            Image.fromarray(img[y:y+TILE, x:x+TILE]).save(img_npy)
            Image.fromarray(tile_ids).save(ids_npy)
            cnt = np.bincount(tile_ids.flatten(), minlength=NUM_FLOODNET_CLASSES)
            records.append((img_npy, ids_npy, split, bool(cnt[1] + cnt[3] + cnt[5] > 0), cnt))
            if split == "train": n_train += 1
            else: n_val += 1
    if (i + 1) % 200 == 0 and tree_bytes(TILES) > MAX_TMP_GB * 1024**3:
        mark("floodnet:tiling-guard", extra=str(i))
        break

rng2 = np.random.default_rng(SEED)
train_recs = [r for r in records if r[2] == "train"]
if len(train_recs) > MAX_TRAIN_TILES:
    flood = [r for r in train_recs if r[3]]
    rest = [r for r in train_recs if not r[3]]
    budget = max(0, MAX_TRAIN_TILES - len(flood))
    keep = set(map(id, flood)) | set(map(id, rng2.choice(rest, size=min(budget, len(rest)), replace=False) if len(rest) else []))
    for r in train_recs:
        if id(r) not in keep:
            r[0].unlink(missing_ok=True); r[1].unlink(missing_ok=True)
    train_recs = [r for r in train_recs if id(r) in keep]
val_recs = [r for r in records if r[2] == "val"]
if len(val_recs) > MAX_VAL_TILES:
    keep = set(map(id, rng2.choice(val_recs, size=MAX_VAL_TILES, replace=False)))
    for r in val_recs:
        if id(r) not in keep:
            r[0].unlink(missing_ok=True); r[1].unlink(missing_ok=True)
    val_recs = [r for r in val_recs if id(r) in keep]

counts = np.sum([r[4] for r in train_recs], axis=0).astype(np.int64) if train_recs else np.zeros(NUM_FLOODNET_CLASSES, dtype=np.int64)
print(f"tiles after cap: train={len(train_recs)} val={len(val_recs)} | tree {tree_bytes(TILES)/1e9:.1f} GB")
print("class pixel counts:", counts.tolist())
assert counts.sum() > 0
mark("floodnet:tiles", extra=f"train={len(train_recs)} val={len(val_recs)}")

In [ ]:

import subprocess as _sp, json as _json
TAR_PATH = WORK / "floodnet_tiles_png.tar"
_sp.run(["tar", "-cf", str(TAR_PATH), "-C", "/tmp", "floodnet_tiles"], check=True)
summary = {
    "train_tiles": len(list((TILES / "train").glob("*_img.png"))),
    "val_tiles": len(list((TILES / "val").glob("*_img.png"))),
    "tar_gb": round(TAR_PATH.stat().st_size / 1e9, 2),
    "class_pixel_counts": counts.tolist(),
}
(WORK / "tilepack_summary.json").write_text(_json.dumps(summary, indent=2))
print(_json.dumps(summary, indent=2))
print("working dir:", sorted(q.name for q in WORK.iterdir()))
mark("tilepack:done", extra=str(summary)[:300])